# 04 · Otros modelos, ensamble y variantes del texto

El modelo del `03_fotos_vendedor_texto.ipynb` (LightGBM) llega a **0,9423 en la validación** y **0,94457 en Kaggle**. Este notebook prueba:

1. **Otros modelos** con las mismas features: **XGBoost** y **CatBoost**.
2. **Ensamble**: promediar las probabilidades de varios modelos.
3. **Dónde se equivoca** el modelo.
4. **Variantes del modelo de texto**, que es la feature más importante.
5. La **combinación** de lo que funcionó, y la submission.

Misma validación que siempre: 5 particiones estratificadas, `random_state=42`. Ruido entre particiones: ~±0,001.

**Cambio de organización:** las funciones de features de los notebooks 02 y 03 ahora están en **`features.py`**, sin cambios en la lógica (verificado: reproduce exacto el CSV del 03). Este notebook y los scripts finales (`train.py` / `predict.py`) las importan de ahí. Así hay una sola versión del código, y lo que se valida es exactamente lo que se entrega.

## 0. Datos, features y piezas

- `armar_tabla`: las 92 features que no usan la respuesta (02 + 03).
- `calcular_piezas`: las 3 que sí la usan (target encoding de vendedor y categoría, probabilidad del texto combinado), calculadas solo con las filas de entrenamiento de cada partición.
- `PARTICIONES`: las 5 de validación y, en la posición 5, train → test.

Tarda unos 7 minutos.

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from features import (cargar_datos, armar_tabla, calcular_piezas, armar_conjunto, texto_combinado,
                      vectorizador_texto, CAT, g)

todo, y = cargar_datos()
N_TRAIN = len(y)
X_todo = armar_tabla(todo)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
PARTICIONES = list(cv.split(np.zeros(N_TRAIN), y)) + [(np.arange(N_TRAIN), np.arange(N_TRAIN, len(todo)))]

inicio = time.time()
piezas = [calcular_piezas(todo, y, idx_a, idx_b) for idx_a, idx_b in PARTICIONES]
print(X_todo.shape, f"piezas listas en {(time.time() - inicio) / 60:.1f} minutos")

## 1. Tres modelos de *gradient boosting*

Los tres construyen árboles de decisión **de a uno**, y cada árbol nuevo corrige los errores de los anteriores. La diferencia está en **cómo arma cada uno sus árboles**:

- **LightGBM** (el del 03): hace crecer el árbol por la hoja que más mejora (*leaf-wise*), así que salen árboles asimétricos. Es rápido.
- **XGBoost**: hace crecer el árbol **nivel por nivel** hasta una profundidad máxima (`max_depth=8`). `subsample=0.8`: cada árbol ve el 80% de las filas.
- **CatBoost**: árboles **simétricos** (la misma pregunta en todo un nivel) y un manejo propio de las categóricas. Suele sobreajustar menos, pero es bastante más lento. Necesita las categóricas como texto, sin vacíos: las categorías que no aparecen en entrenamiento (que `alinear_categorias` deja vacías) pasan a `"NA"`.

Todos con las mismas 95 columnas y la misma semilla. Para cada uno se guarda la **probabilidad de usado** de cada ítem de validación (`oof`, *out of fold*), que se usa después para el ensamble.

Tarda unos 25 minutos (CatBoost, 15).

In [ ]:
PARAMS_LGBM = dict(n_estimators=1500, learning_rate=0.02, num_leaves=63, colsample_bytree=0.5, random_state=42, verbose=-1)
PARAMS_XGB  = dict(n_estimators=1500, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.5,
                   min_child_weight=2, tree_method="hist", enable_categorical=True, random_state=42, n_jobs=8)
PARAMS_CAT  = dict(iterations=1500, learning_rate=0.08, depth=6, random_seed=42, verbose=0, thread_count=8)

def categoricas_como_texto(A, B):
    A, B = A.copy(), B.copy()
    for c in CAT:
        A[c] = A[c].astype(object).fillna("NA").astype(str)
        B[c] = B[c].astype(object).fillna("NA").astype(str)
    return A, B

def entrenar_y_predecir(nombre, A, yA, B):
    if nombre == "lgbm":
        return LGBMClassifier(**PARAMS_LGBM).fit(A, yA).predict_proba(B)[:, 1]
    if nombre == "xgb":
        return XGBClassifier(**PARAMS_XGB).fit(A, yA).predict_proba(B)[:, 1]
    A, B = categoricas_como_texto(A, B)
    return CatBoostClassifier(**PARAMS_CAT).fit(A, yA, cat_features=CAT).predict_proba(B)[:, 1]

def probabilidades_oof(nombre, piezas_por_particion):
    p = np.zeros(N_TRAIN)
    for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
        A, B = armar_conjunto(X_todo, piezas_por_particion[f], idx_a, idx_b)
        p[idx_b] = entrenar_y_predecir(nombre, A, y[idx_a], B)
    return p

def accuracy_por_particion(p):
    return np.array([accuracy_score(y[idx_b], p[idx_b] > 0.5) for _, idx_b in PARTICIONES[:5]])

oof = {}
for nombre in ["lgbm", "xgb", "cat"]:
    inicio = time.time()
    oof[nombre] = probabilidades_oof(nombre, piezas)
    notas = accuracy_por_particion(oof[nombre])
    print(f"{nombre:5s} {notas.mean():.4f} ± {notas.std():.4f}   ({(time.time() - inicio) / 60:.1f} min)")

## 2. Ensamble: promediar modelos

Si dos modelos se equivocan en ítems **distintos**, promediar sus probabilidades corrige parte de los errores: donde uno duda y el otro está seguro, gana el seguro.

Para que sirva, los modelos tienen que ser **distintos**. Primero se mide cuánto coinciden:

In [ ]:
from itertools import combinations

for a, b in combinations(oof, 2):
    desacuerdo = ((oof[a] > 0.5) != (oof[b] > 0.5)).mean()
    print(f"{a} vs {b}: no coinciden en el {desacuerdo:.1%} de los ítems  (correlación {np.corrcoef(oof[a], oof[b])[0, 1]:.4f})")

Coinciden en el 98,5% al 99% de los ítems: aprenden casi lo mismo, porque usan las mismas features. El margen del ensamble es chico.

Para distinguir una mejora real del ruido, se mira la diferencia **en cada partición** contra LightGBM solo. Si es ruido, lo esperable es que mejore en algunas y empeore en otras. Si mejora en **las 5**, es más creíble.

In [ ]:
ref = accuracy_por_particion(oof["lgbm"])
resultados = []

def comparar(nombre, p):
    notas = accuracy_por_particion(p)
    dif = notas - ref
    resultados.append({"modelo": nombre, "accuracy": round(notas.mean(), 4), "mejora en": f"{(dif > 0).sum()}/5",
                       "diferencia por partición": " ".join(f"{d:+.4f}" for d in dif)})

comparar("LightGBM (modelo del 03)", oof["lgbm"])
comparar("XGBoost", oof["xgb"])
comparar("CatBoost", oof["cat"])
comparar("promedio LightGBM + XGBoost", (oof["lgbm"] + oof["xgb"]) / 2)
comparar("promedio LightGBM + CatBoost", (oof["lgbm"] + oof["cat"]) / 2)
comparar("promedio de los 3", (oof["lgbm"] + oof["xgb"] + oof["cat"]) / 3)
pd.DataFrame(resultados)

**Lectura:**

- **XGBoost** queda apenas por encima de LightGBM y **CatBoost** apenas por debajo. Todo dentro del ruido: con estas features, el algoritmo importa poco.
- **LightGBM + XGBoost** mejora poco (+0,0006), pero lo hace en las 5 particiones. Es la mejor combinación.
- Sumar CatBoost no agrega nada y es el más lento. Se descarta.

## 3. ¿Dónde se equivoca el modelo?

Error de LightGBM según cuántos ítems publica el vendedor (contando train + test):

In [ ]:
errores = (oof["lgbm"] > 0.5) != y
n_items = X_todo["vendedor_n_items"].iloc[:N_TRAIN].values
grupos = pd.cut(n_items, [0, 1, 5, 50, np.inf], labels=["1 ítem", "2 a 5", "6 a 50", "más de 50"])
tabla = pd.DataFrame({"grupo": grupos, "error": errores}).groupby("grupo", observed=True)["error"].agg(["mean", "size", "sum"])
tabla.columns = ["tasa de error", "ítems", "errores"]
tabla["% de todos los errores"] = tabla["errores"] / tabla["errores"].sum()
tabla.round(3)

**Lectura:** el modelo se equivoca **5 veces más** con vendedores de un solo ítem que con los grandes, y ese grupo junta casi la mitad de los errores. Es lógico: sin historial del vendedor, el *target encoding* no ayuda y el modelo solo puede mirar el ítem (texto, fotos, precio).

Es la principal **limitación** del modelo. La forma de mejorar en ese grupo es sacarle más jugo al ítem, y la feature más fuerte del ítem es el **texto**.

## 4. Variantes del modelo de texto

Sobre el texto combinado (título + garantía + atributos) del 03, se prueban cuatro variantes. Cada una reemplaza a `prob_combinado` y se evalúa con LightGBM:

- **v1 · tokens estructurados:** se agregan al texto "palabras" artificiales con datos de la publicación: `cat_MLA1234` (categoría), `tipo_gold`, `foto_1200x900`, `pago_MLAWC`, `envio_me2`, `st_paused`, `cant_1`. Así la regresión logística combina palabras con contexto.
- **v2 · `C=2`** y **v3 · `C=8`:** `C` controla la regularización de la logística. Más chico = modelo más simple; más grande = se ajusta más a los datos. El del 03 usa `C=4`.
- **v4 · palabras + caracteres:** al TF-IDF de palabras se le suma otro de **pedazos de 3 a 5 letras** (`char_wb`). Reconoce raíces y variantes ("usado", "usada", "usadito", "usdo") que el de palabras ve como palabras distintas.

En el 03 se había probado un modelo **separado** de caracteres solo con el título, y no sumaba. Acá es distinto: palabras y caracteres van en **el mismo** modelo, sobre el texto combinado.

Tarda unos 30 minutos (v4 es la más lenta).

In [ ]:
def tokens_estructurados(df):
    foto = df["pictures"].apply(lambda L: "foto_" + L[0]["max_size"] if L else "foto_no")
    pagos = df["non_mercado_pago_payment_methods"].apply(lambda L: " ".join("pago_" + d["id"] for d in L))
    return ("cat_" + df["category_id"] + " tipo_" + df["listing_type_id"] + " " + foto + " " + pagos
            + " envio_" + df["shipping"].apply(lambda s: str(g(s, "mode"))) + " st_" + df["status"]
            + " cant_" + df["initial_quantity"].clip(upper=10).astype(str))

def prob_texto_variante(textos_a, y_a, textos_b, caracteres=False, C=4):
    modelo = make_pipeline(vectorizador_texto(caracteres), LogisticRegression(max_iter=3000, C=C))
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    p_a = cross_val_predict(modelo, textos_a, y_a, cv=kf, method="predict_proba")[:, 1]
    return p_a, modelo.fit(textos_a, y_a).predict_proba(textos_b)[:, 1]

texto = texto_combinado(todo)
VARIANTES = {
    "v1 tokens estructurados": dict(textos=texto + " || " + tokens_estructurados(todo)),
    "v2 C=2":                  dict(textos=texto, C=2),
    "v3 C=8":                  dict(textos=texto, C=8),
    "v4 palabras + caracteres": dict(textos=texto, caracteres=True),
}

piezas_variante, oof_texto = {}, {}
for nombre, v in VARIANTES.items():
    inicio = time.time()
    t = v["textos"]
    piezas_variante[nombre] = []
    p_texto = np.zeros(N_TRAIN)
    for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
        pz = dict(piezas[f])
        pz["prob_combinado"] = prob_texto_variante(t.iloc[idx_a].reset_index(drop=True), y[idx_a],
                                                   t.iloc[idx_b].reset_index(drop=True),
                                                   v.get("caracteres", False), v.get("C", 4))
        piezas_variante[nombre].append(pz)
        p_texto[idx_b] = pz["prob_combinado"][1]
    oof_texto[nombre] = p_texto
    print(f"{nombre} lista ({(time.time() - inicio) / 60:.1f} min)")

In [ ]:
p_texto_03 = np.zeros(N_TRAIN)
for f, (_, idx_b) in enumerate(PARTICIONES[:5]):
    p_texto_03[idx_b] = piezas[f]["prob_combinado"][1]

filas = [{"variante": "texto del 03 (palabras, C=4)", "texto solo": accuracy_por_particion(p_texto_03).mean(),
          "modelo completo (LightGBM)": accuracy_por_particion(oof["lgbm"]).mean()}]
for nombre in VARIANTES:
    p = probabilidades_oof("lgbm", piezas_variante[nombre])
    filas.append({"variante": nombre, "texto solo": accuracy_por_particion(oof_texto[nombre]).mean(),
                  "modelo completo (LightGBM)": accuracy_por_particion(p).mean()})
pd.DataFrame(filas).round(4)

**Lectura:**

- **v1** es la lección más interesante. El modelo de texto **solo** mejora muchísimo (de ~0,84 a ~0,91), pero el modelo completo **empeora**. Los datos que se agregaron al texto (tipo de publicación, fotos, pagos) LightGBM ya los tiene como columnas. Entonces la probabilidad del texto pasa a repetir esa información en vez de aportar algo nuevo. **Lo que vale de una feature es la información nueva que agrega, no qué tan buena es sola.**
- **v2 y v3**: cambiar `C` no mueve nada. `C=4` estaba bien.
- **v4** mejora un poco el texto solo y el modelo completo. Es la única variante que suma.

## 5. Combinación: texto v4 + LightGBM + XGBoost

Las dos mejoras chicas que quedaron en pie (texto v4 y promedio con XGBoost) son independientes: una cambia una feature y la otra, el modelo. Se prueban juntas, siempre partición por partición contra el modelo del 03.

In [ ]:
oof_v4 = {nombre: probabilidades_oof(nombre, piezas_variante["v4 palabras + caracteres"]) for nombre in ["lgbm", "xgb"]}

comparar("LightGBM, texto v4", oof_v4["lgbm"])
comparar("XGBoost, texto v4", oof_v4["xgb"])
comparar("promedio LightGBM + XGBoost, texto v4", (oof_v4["lgbm"] + oof_v4["xgb"]) / 2)
pd.DataFrame(resultados)

**Lectura:** la combinación **texto v4 + promedio LightGBM/XGBoost** es la mejor y **mejora en las 5 particiones**. Son unas 70 predicciones más acertadas de 70.000. Es poco, pero consistente.

**¿Y el umbral?** Se predice "usado" cuando la probabilidad es mayor a 0,5. Se puede buscar otro umbral que maximice la accuracy:

In [ ]:
p_final = (oof_v4["lgbm"] + oof_v4["xgb"]) / 2
umbrales = np.round(np.arange(0.40, 0.605, 0.01), 2)
acc = pd.Series([accuracy_score(y, p_final > u) for u in umbrales], index=umbrales)
print(f"mejor umbral: {acc.idxmax()} -> {acc.max():.4f}   |   umbral 0,5 -> {acc[0.5]:.4f}")

El mejor umbral gana +0,0002: ruido, y además estaría elegido mirando las mismas notas con las que se evalúa. **Se mantiene 0,5.**

## 6. Modelo final y submission

1. Piezas con todo train → test, con el texto v4 (`calcular_piezas(..., caracteres=True)`, igual que lo va a hacer `train.py`).
2. LightGBM y XGBoost entrenados con los 70.000 ítems.
3. Se promedian las dos probabilidades y se predice "usado" si el promedio es mayor a 0,5.

In [ ]:
import os

idx_a, idx_b = PARTICIONES[5]
piezas_final = calcular_piezas(todo, y, idx_a, idx_b, caracteres=True)
A, B = armar_conjunto(X_todo, piezas_final, idx_a, idx_b)

p_lgbm = LGBMClassifier(**PARAMS_LGBM).fit(A, y).predict_proba(B)[:, 1]
p_xgb  = XGBClassifier(**PARAMS_XGB).fit(A, y).predict_proba(B)[:, 1]
p_test = (p_lgbm + p_xgb) / 2

sub = pd.DataFrame({"ID": range(1, len(B) + 1), "condition": np.where(p_test > 0.5, "used", "new")})
os.makedirs("submissions", exist_ok=True)
sub.to_csv("submissions/04_lgbm_xgb_texto_v4.csv", index=False)

anterior = pd.read_csv("submissions/03_fotos_vendedor_texto.csv")
print(A.shape, sub.shape)
print(sub["condition"].value_counts(normalize=True).round(3))
print(f"cambian {(sub['condition'] != anterior['condition']).sum()} predicciones respecto de la submission 03")